### Cốt lõi của thuật toán (UCS)

In [1]:
import heapq
from itertools import count

class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

class Problem:
    def __init__(self, start_state, target_state, get_moves, apply_move, step_cost):
        self.start_state = start_state
        self.target_state = target_state
        self.get_moves = get_moves
        self.apply_move = apply_move
        self.step_cost = step_cost

def is_goal(puzzle, cur_state):
    if callable(puzzle.target_state):
        return puzzle.target_state(cur_state)
    return cur_state == puzzle.target_state

arrival = count()

def PriorityQueue():
    return []

def enqueue(wait_list, item):
    heapq.heappush(wait_list, (item.path_cost, next(arrival), item))

def is_empty(wait_list):
    return len(wait_list) == 0

def dequeue(wait_list):
    if is_empty(wait_list):
        return None
    return heapq.heappop(wait_list)[2]

def find_in_queue(wait_list, state):
    for idx in range(len(wait_list)):
        if wait_list[idx][2].state == state:
            return idx
    return -1

def replace(wait_list, idx, item):
    wait_list[idx] = (item.path_cost, next(arrival), item)
    heapq.heapify(wait_list)

def child_node(puzzle, prev_node, move):
    next_state = puzzle.apply_move(prev_node.state, move)
    next_cost = prev_node.path_cost + puzzle.step_cost(prev_node.state, move)
    return Node(state=next_state, parent=prev_node, action=move, path_cost=next_cost)

def solution(goal_node):
    trace = []
    curr = goal_node
    while curr is not None:
        trace.append((curr.state, curr.path_cost))
        curr = curr.parent
    trace.reverse()
    return trace

def UCS(puzzle):
    root = Node(puzzle.start_state)

    wait_queue = PriorityQueue()
    enqueue(wait_queue, root)
    explored = set()

    while not is_empty(wait_queue):
        cur_node = dequeue(wait_queue)

        if is_goal(puzzle, cur_node.state):
            return solution(cur_node)

        explored.add(cur_node.state)

        for move in puzzle.get_moves(cur_node.state):
            next_node = child_node(puzzle, cur_node, move)
            pos = find_in_queue(wait_queue, next_node.state)

            if next_node.state not in explored and pos == -1:
                enqueue(wait_queue, next_node)
            elif pos != -1 and wait_queue[pos][2].path_cost > next_node.path_cost:
                replace(wait_queue, pos, next_node)
    return None

### Bài toán 8-Puzzle

In [2]:
def find_blank(cur_board):
    for row_idx in range(3):
        for col_idx in range(3):
            if cur_board[row_idx][col_idx] == 0:
                return row_idx, col_idx

def actions_8puzzle(cur_board):
    blank_r, blank_c = find_blank(cur_board)
    valid_moves = []

    if blank_r > 0: valid_moves.append("Lên")
    if blank_r < 2: valid_moves.append("Xuống")
    if blank_c > 0: valid_moves.append("Trái")
    if blank_c < 2: valid_moves.append("Phải")

    return valid_moves

def result_8puzzle(cur_board, move):
    blank_r, blank_c = find_blank(cur_board)

    new_r, new_c = blank_r, blank_c
    if move == "Lên":
        new_r -= 1
    elif move == "Xuống":
        new_r += 1
    elif move == "Trái":
        new_c -= 1
    elif move == "Phải":
        new_c += 1

    grid = [list(cur_board[0]), list(cur_board[1]), list(cur_board[2])]
    grid[blank_r][blank_c] = grid[new_r][new_c]
    grid[new_r][new_c] = 0

    return (tuple(grid[0]), tuple(grid[1]), tuple(grid[2]))

def cost_8puzzle(cur_board, move):
    if move in ("Lên", "Xuống"):
        return 1
    return 2

def print_state(board):
    for r in range(3):
        print(f"[{board[r][0]} {board[r][1]} {board[r][2]}]")
    print()

if __name__ == "__main__":
    start_board = ((1, 2, 3), (4, 0, 6), (7, 5, 8))
    target_board = ((1, 2, 3), (4, 5, 6), (7, 8, 0))
    puzzle = Problem(start_board, target_board, actions_8puzzle, result_8puzzle, cost_8puzzle)

    steps = UCS(puzzle)
    if steps is None:
        print("Không tìm thấy lời giải")
    else:
        print(f"SOLUTION (Tổng chi phí: {steps[-1][1]})")
        for step_idx in range(len(steps)):
            board, cost = steps[step_idx]
            if step_idx == len(steps) - 1:
                print(f"Bước {step_idx} (GOAL), g = {cost}:")
            else:
                print(f"Bước {step_idx}, g = {cost}:")
            print_state(board)

SOLUTION (Tổng chi phí: 3)
Bước 0, g = 0:
[1 2 3]
[4 0 6]
[7 5 8]

Bước 1, g = 1:
[1 2 3]
[4 5 6]
[7 0 8]

Bước 2 (GOAL), g = 3:
[1 2 3]
[4 5 6]
[7 8 0]


### Bài toán Vacuum Robot

In [3]:
def actions_vacuum(cur_state):
    return ["Hút", "Trái", "Phải"]

def result_vacuum(cur_state, move):
    pos, room_a, room_b = cur_state

    if move == "Trái":
        return ("A", room_a, room_b)
    elif move == "Phải":
        return ("B", room_a, room_b)
    elif move == "Hút":
        if pos == "A":
            return (pos, "Sạch", room_b)
        else:
            return (pos, room_a, "Sạch")

def cost_vacuum(cur_state, move):
    if move == "Hút":
        return 1
    return 2

def goal_vacuum(cur_state):
    return cur_state[1] == "Sạch" and cur_state[2] == "Sạch"

def print_vacuum_state(cur_state):
    pos, room_a, room_b = cur_state
    print(f"Robot ở phòng {pos} | Phòng A: {room_a} | Phòng B: {room_b}")

if __name__ == "__main__":
    start_vacuum = ("A", "Bẩn", "Bẩn")
    vacuum_puzzle = Problem(start_vacuum, goal_vacuum, actions_vacuum, result_vacuum, cost_vacuum)

    steps = UCS(vacuum_puzzle)
    if steps is None:
        print("Không tìm thấy lời giải")
    else:
        print(f"SOLUTION (Tổng chi phí: {steps[-1][1]})")
        for step_idx in range(len(steps)):
            state, cost = steps[step_idx]
            if step_idx == len(steps) - 1:
                print(f"Bước {step_idx} (GOAL), g = {cost}:")
            else:
                print(f"Bước {step_idx}, g = {cost}:")
            print_vacuum_state(state)

SOLUTION (Tổng chi phí: 4)
Bước 0, g = 0:
Robot ở phòng A | Phòng A: Bẩn | Phòng B: Bẩn
Bước 1, g = 1:
Robot ở phòng A | Phòng A: Sạch | Phòng B: Bẩn
Bước 2, g = 3:
Robot ở phòng B | Phòng A: Sạch | Phòng B: Bẩn
Bước 3 (GOAL), g = 4:
Robot ở phòng B | Phòng A: Sạch | Phòng B: Sạch


### UCS bản tối ưu (nhanh hơn)

In [4]:
import time

def UCS_nhanh(puzzle):
    root = Node(puzzle.start_state)

    wait_queue = PriorityQueue()
    enqueue(wait_queue, root)
    best_cost = {root.state: root.path_cost}
    explored = set()

    while not is_empty(wait_queue):
        cur_node = dequeue(wait_queue)

        if cur_node.state in explored:
            continue

        if is_goal(puzzle, cur_node.state):
            return solution(cur_node)

        explored.add(cur_node.state)

        for move in puzzle.get_moves(cur_node.state):
            next_node = child_node(puzzle, cur_node, move)

            if next_node.state in explored:
                continue
            if next_node.state not in best_cost or next_node.path_cost < best_cost[next_node.state]:
                best_cost[next_node.state] = next_node.path_cost
                enqueue(wait_queue, next_node)
    return None

if __name__ == "__main__":
    samples = [
        ("8-Puzzle", Problem(((1, 2, 3), (4, 0, 6), (7, 5, 8)), ((1, 2, 3), (4, 5, 6), (7, 8, 0)),
                             actions_8puzzle, result_8puzzle, cost_8puzzle)),
        ("Vacuum", Problem(("A", "Bẩn", "Bẩn"), goal_vacuum,
                           actions_vacuum, result_vacuum, cost_vacuum)),
    ]
    for name, prob in samples:
        slow = UCS(prob)[-1][1]
        fast = UCS_nhanh(prob)[-1][1]
        print(f"{name}: UCS = {slow}, UCS_nhanh = {fast}, {'cùng chi phí' if slow == fast else 'KHÁC NHAU'}")

    hard_start = ((1, 0, 8), (4, 5, 2), (6, 7, 3))
    hard_target = ((1, 2, 3), (4, 5, 6), (7, 8, 0))
    hard_puzzle = Problem(hard_start, hard_target, actions_8puzzle, result_8puzzle, cost_8puzzle)

    t0 = time.time()
    steps = UCS_nhanh(hard_puzzle)
    print(f"Đề khó: UCS_nhanh tìm được lời giải {len(steps) - 1} bước, chi phí {steps[-1][1]}, mất {time.time() - t0:.1f} giây")

8-Puzzle: UCS = 3, UCS_nhanh = 3, cùng chi phí
Vacuum: UCS = 4, UCS_nhanh = 4, cùng chi phí
Đề khó: UCS_nhanh tìm được lời giải 21 bước, chi phí 28, mất 1.1 giây
